# 🐋 HyperWhale API Playground

Test Hyperliquid's public API interactively. No API key needed — all endpoints are read-only.

**How it works:** Every call is a POST to `https://api.hyperliquid.xyz/info` with a JSON body. The `"type"` field tells the API what you want.

In [1]:
# Setup — add our src folder to Python's path so we can import hyperwhale
import sys
sys.path.insert(0, r"C:\Users\Sahil\HyperLiquid\src")

import json
import httpx
from datetime import datetime, timedelta, timezone

# The one URL we'll use for everything
API_URL = "https://api.hyperliquid.xyz/info"

# Reusable HTTP client (keeps connection alive)
client = httpx.Client(timeout=30.0)

def api_call(payload: dict) -> dict | list:
    """Make a POST to Hyperliquid and return parsed JSON."""
    resp = client.post(API_URL, json=payload, headers={"Content-Type": "application/json"})
    resp.raise_for_status()
    return resp.json()

print("✅ Ready — client connected to", API_URL)

✅ Ready — client connected to https://api.hyperliquid.xyz/info


## 1. Market Data — Get All Mid Prices

The simplest call. Returns mid prices for all ~520 listed assets. No wallet address needed.

In [2]:
# Fetch all mid prices
mids = api_call({"type": "allMids"})

print(f"📊 {len(mids)} assets listed on Hyperliquid\n")
print(f"  BTC:  ${float(mids['BTC']):>12,.2f}")
print(f"  ETH:  ${float(mids['ETH']):>12,.2f}")
print(f"  HYPE: ${float(mids.get('HYPE', 0)):>12,.2f}")
print(f"  SOL:  ${float(mids.get('SOL', 0)):>12,.2f}")

# Top 10 by price
print("\n--- Top 10 by price ---")
sorted_mids = sorted(mids.items(), key=lambda x: float(x[1]), reverse=True)
for coin, price in sorted_mids[:10]:
    print(f"  {coin:>8s}: ${float(price):>12,.2f}")

📊 521 assets listed on Hyperliquid

  BTC:  $   67,911.50
  ETH:  $    1,954.65
  HYPE: $       29.58
  SOL:  $       83.96

--- Top 10 by price ---
      @234: $   67,920.50
       BTC: $   67,911.50
      @142: $   67,908.50
      @173: $   38,908.50
      PAXG: $    5,039.75
      @182: $    5,014.10
      @209: $    5,013.95
   PANDORA: $    3,026.30
      @151: $    1,955.35
      @235: $    1,955.35


## 2. Fetch a Whale's Positions — `clearinghouseState`

This is the most important call. Given a wallet address, returns their full portfolio: account value, margin, and every open position.

**Try changing the address to any wallet you want to investigate.**

In [3]:
# 🔧 CHANGE THIS to any wallet address you want to explore
WHALE_ADDRESS = "0xefffa330cbae8d916ad1d33f0eeb16cfa711fa91"

# Fetch their full portfolio state
raw = api_call({"type": "clearinghouseState", "user": WHALE_ADDRESS})

# --- Account Summary ---
margin = raw.get("marginSummary", {})
print(f"🐋 Wallet: {WHALE_ADDRESS[:10]}...{WHALE_ADDRESS[-4:]}")
print(f"   Account Value:  ${float(margin.get('accountValue', 0)):>14,.2f}")
print(f"   Margin Used:    ${float(margin.get('totalMarginUsed', 0)):>14,.2f}")
print(f"   Total Notional: ${float(margin.get('totalNtlPos', 0)):>14,.2f}")
print(f"   Withdrawable:   ${float(raw.get('withdrawable', 0)):>14,.2f}")

# --- Open Positions ---
positions = raw.get("assetPositions", [])
print(f"\n   Open Positions: {len(positions)}")
print(f"   {'Coin':>8s}  {'Side':>6s}  {'Size':>12s}  {'Notional':>14s}  {'PnL':>12s}  {'Leverage':>8s}")
print("   " + "-" * 72)

for ap in positions:
    pos = ap.get("position", {})
    szi = float(pos.get("szi", 0))
    if szi == 0:
        continue
    side = "LONG" if szi > 0 else "SHORT"
    notional = abs(float(pos.get("positionValue", 0)))
    pnl = float(pos.get("unrealizedPnl", 0))
    lev = pos.get("leverage", {})
    lev_str = f"{float(lev.get('value', 1)):.0f}x {lev.get('type', 'cross')}"
    print(f"   {pos.get('coin', '?'):>8s}  {side:>6s}  {abs(szi):>12.4f}  ${notional:>13,.2f}  ${pnl:>11,.2f}  {lev_str:>8s}")

🐋 Wallet: 0xefffa330...fa91
   Account Value:  $  4,427,613.94
   Margin Used:    $  1,105,020.00
   Total Notional: $ 11,050,200.00
   Withdrawable:   $  3,322,593.94

   Open Positions: 1
       Coin    Side          Size        Notional           PnL  Leverage
   ------------------------------------------------------------------------
       AVAX    LONG  1200000.0000  $11,050,200.00  $  83,862.99  10x cross


In [4]:
# 👀 See the FULL raw JSON response (expand to explore the structure)
print(json.dumps(raw, indent=2)[:3000])  # first 3000 chars to avoid flooding

{
  "marginSummary": {
    "accountValue": "4427613.9387609996",
    "totalNtlPos": "11050200.0",
    "totalRawUsd": "-6622586.0612390004",
    "totalMarginUsed": "1105020.0"
  },
  "crossMarginSummary": {
    "accountValue": "4427613.9387609996",
    "totalNtlPos": "11050200.0",
    "totalRawUsd": "-6622586.0612390004",
    "totalMarginUsed": "1105020.0"
  },
  "crossMaintenanceMarginUsed": "552510.0",
  "withdrawable": "3322593.9387610001",
  "assetPositions": [
    {
      "type": "oneWay",
      "position": {
        "coin": "AVAX",
        "szi": "1200000.0",
        "leverage": {
          "type": "cross",
          "value": 10
        },
        "entryPx": "9.1386",
        "positionValue": "11050200.0",
        "unrealizedPnl": "83862.985943",
        "returnOnEquity": "0.0764731066",
        "liquidationPx": "5.8092860186",
        "marginUsed": "1105020.0",
        "maxLeverage": 10,
        "cumFunding": {
          "allTime": "99793.856992",
          "sinceOpen": "36733.17

## 3. Fetch Trade History — `userFillsByTime`

Get the last 30 days of trades for a wallet. The API returns up to 2000 fills.
This is what powers the **activity score** in our scoring system.

In [5]:
# Fetch fills from the last 30 days
start_ms = int((datetime.now(timezone.utc) - timedelta(days=30)).timestamp() * 1000)

fills = api_call({
    "type": "userFillsByTime",
    "user": WHALE_ADDRESS,
    "startTime": start_ms,
})

print(f"📜 {len(fills)} trades in the last 30 days\n")

if fills:
    # Show the 10 most recent
    print(f"  {'Time':>20s}  {'Coin':>8s}  {'Dir':>12s}  {'Price':>12s}  {'Size':>10s}  {'PnL':>10s}")
    print("  " + "-" * 80)
    for f in fills[-10:]:
        ts = datetime.fromtimestamp(f["time"] / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M")
        print(f"  {ts:>20s}  {f.get('coin', '?'):>8s}  {f.get('dir', '?'):>12s}  ${float(f.get('px', 0)):>11,.2f}  {float(f.get('sz', 0)):>10.4f}  ${float(f.get('closedPnl', 0)):>9,.2f}")

    # What coins do they trade?
    coins = set(f.get("coin", "") for f in fills)
    print(f"\n  Coins traded: {', '.join(sorted(coins))}")

    # Show one raw fill so you can see ALL the fields
    print(f"\n--- Raw fill example ---")
    print(json.dumps(fills[-1], indent=2))

📜 2000 trades in the last 30 days

                  Time      Coin           Dir         Price        Size         PnL
  --------------------------------------------------------------------------------
      2026-01-23 18:10       ZRO    Open Short  $       2.32    631.4000  $     0.00
      2026-01-23 18:10       ZRO    Open Short  $       2.32   1066.9000  $     0.00
      2026-01-23 18:10       ZRO    Open Short  $       2.32    631.4000  $     0.00
      2026-01-23 18:10       ZRO    Open Short  $       2.32   1819.9000  $     0.00
      2026-01-23 18:10       ZRO    Open Short  $       2.32    272.0000  $     0.00
      2026-01-23 18:10       ZRO    Open Short  $       2.32    299.9000  $     0.00
      2026-01-23 18:10       ZRO    Open Short  $       2.32    631.4000  $     0.00
      2026-01-23 18:10       ZRO    Open Short  $       2.32    158.1000  $     0.00
      2026-01-23 18:10       ZRO    Open Short  $       2.32    572.6000  $     0.00
      2026-01-23 18:10       ZRO

## 4. Use Our Collector Class — the clean way

Instead of raw `api_call()`, use the `HyperliquidCollector` from our codebase.
It does the same API calls but returns **typed models** instead of raw dicts.

In [6]:
# Our collector wraps the same API but gives you typed Python objects
from hyperwhale.data.collector import HyperliquidCollector

collector = HyperliquidCollector()

# fetch_position_snapshot() = clearinghouseState + parsing
snapshot = collector.fetch_position_snapshot(WHALE_ADDRESS)

# Now you get clean Python objects instead of messy dicts
print(f"Account Value:  ${snapshot.account_value:,.2f}")
print(f"Margin Used:    ${snapshot.total_margin_used:,.2f}")
print(f"Positions:      {len(snapshot.positions)}\n")

for p in snapshot.positions:
    print(f"  {p.coin:>8s}  {p.side.value:>5s}  size={p.size:.4f}  "
          f"notional=${p.notional_value:,.0f}  pnl=${p.unrealized_pnl:,.2f}  "
          f"lev={p.leverage:.0f}x")

# Notice: p.side is a PositionSide enum, not a string
# p.side == PositionSide.LONG → True
# p.side.value → "long"

Account Value:  $4,429,293.94
Margin Used:    $1,105,188.00
Positions:      1

      AVAX   long  size=1200000.0000  notional=$11,051,880  pnl=$85,542.99  lev=10x


## 5. Score a Whale — see the scoring system in action

Take the data we just fetched and run it through our `WhaleScorer`.

In [7]:
from hyperwhale.scoring import WhaleScorer

scorer = WhaleScorer()

# Use the live data we already fetched
total_notional = sum(p.notional_value for p in snapshot.positions)
trade_count = len(fills)  # from cell 3

result = scorer.score(
    account_value=snapshot.account_value,
    total_notional=total_notional,
    trade_count_30d=trade_count,
)

print(f"🎯 Scoring Results for {WHALE_ADDRESS[:10]}...")
print(f"")
print(f"   Tier:            {result.tier.value.upper()}")
print(f"   Composite Score: {result.whale_score:.1f} / 100")
print(f"")
print(f"   Account Score:   {result.account_score:.0f}  (AV = ${snapshot.account_value:,.0f})")
print(f"   Position Score:  {result.position_score:.0f}  (Notional = ${total_notional:,.0f})")
print(f"   Activity Score:  {result.activity_score:.0f}  ({trade_count} trades in 30d)")
print(f"")
print(f"   Formula: 0.60×{result.account_score:.0f} + 0.20×{result.position_score:.0f} + 0.20×{result.activity_score:.0f} = {result.whale_score:.1f}")

🎯 Scoring Results for 0xefffa330...

   Tier:            SHARK
   Composite Score: 52.0 / 100

   Account Score:   30  (AV = $4,429,294)
   Position Score:  70  (Notional = $11,051,880)
   Activity Score:  100  (2000 trades in 30d)

   Formula: 0.60×30 + 0.20×70 + 0.20×100 = 52.0


## 6. Explore Any API Call — Your sandbox

Try any Hyperliquid API call by changing the payload below. Some ideas:

```python
# Open orders
{"type": "openOrders", "user": "0x..."}

# Historical orders (up to 2000)
{"type": "historicalOrders", "user": "0x..."}

# Vault details (HLP vault)
{"type": "vaultDetails", "vaultAddress": "0xdfc24b077bc1425ad1dea75bcb6f8158e10df303"}

# L2 order book for BTC
{"type": "l2Book", "coin": "BTC", "nSigFigs": 5}

# Candle data (1h candles for ETH)
{"type": "candleSnapshot", "req": {"coin": "ETH", "interval": "1h", "startTime": 0}}

# Funding rates
{"type": "predictedFundings"}

# Asset metadata + contexts (mark price, OI, funding for ALL coins)
{"type": "metaAndAssetCtxs"}
```

In [8]:
# 🔧 CHANGE THIS PAYLOAD to try any API call
payload = {"type": "predictedFundings"}

result = api_call(payload)

# Pretty-print the response (truncated to avoid flooding)
output = json.dumps(result, indent=2)
if len(output) > 5000:
    print(output[:5000])
    print(f"\n... ({len(output)} total chars, truncated)")
else:
    print(output)

[
  [
    "0G",
    [
      [
        "BinPerp",
        {
          "fundingRate": "-0.00093379",
          "nextFundingTime": 1771588800000,
          "fundingIntervalHours": 4
        }
      ],
      [
        "HlPerp",
        {
          "fundingRate": "-0.0001504169",
          "nextFundingTime": 1771585200000,
          "fundingIntervalHours": 1
        }
      ],
      [
        "BybitPerp",
        {
          "fundingRate": "0.0000125",
          "nextFundingTime": 1771588800000,
          "fundingIntervalHours": 1
        }
      ]
    ]
  ],
  [
    "2Z",
    [
      [
        "BinPerp",
        {
          "fundingRate": "-0.00007346",
          "nextFundingTime": 1771588800000,
          "fundingIntervalHours": 4
        }
      ],
      [
        "HlPerp",
        {
          "fundingRate": "0.0000125",
          "nextFundingTime": 1771585200000,
          "fundingIntervalHours": 1
        }
      ],
      [
        "BybitPerp",
        {
          "fundingRate": "0.000

In [9]:
# 🧹 Cleanup — close the HTTP client when done
client.close()
collector.close()
print("Connections closed.")

Connections closed.
